# SageMath 10.5 Script für eine rotierende 3D-Animation der Kepler-Packung (FCC)

## Fokus: Visualisierung der Oktaederlücken

In [1]:
import numpy as np

In [2]:
def create_fcc_animation():
    # 1. Parameter definieren
    a = 1.0  # Kantenlänge der Elementarzelle
    # Radius der Packungskugel (Atome berühren sich auf der Flächendiagonalen: 4r = a*sqrt(2))
    r_atom = (a * sqrt(2)) / 4
    # Radius der Oktaederlücke (R = r * (sqrt(2) - 1))
    r_void = r_atom * (sqrt(2) - 1)
    
    # Farben
    color_atom = "silver"
    color_void = "red"
    color_cage = "gold"

    # Listen für die Grafikobjekte
    atoms = []
    voids = []
    
    # 2. Gitter generieren (2x2x2 Elementarzellen für einen guten Ausschnitt)
    # Wir iterieren durch Koordinaten und prüfen auf FCC Bedingung
    # FCC Basis in kartesischen Koordinaten (normiert auf a=1):
    # Knoten sind bei (x,y,z) wenn x,y,z ganzzahlig oder alle halbzahlig (im 2er Schritt) sind.
    
    # Einfacherer Ansatz: Wir nutzen die Standard-Basisvektoren für FCC
    # Atome an Ecken (0,0,0) und Flächenmitten (0.5, 0.5, 0) etc.
    
    range_cells = 2 # Anzahl der Zellen pro Achse
    
    all_objects = Graphics()

    # --- ATOME GENERIEREN ---
    for x in range(range_cells + 1):
        for y in range(range_cells + 1):
            for z in range(range_cells + 1):
                # Eckpunkte (Corner atoms)
                atoms.append(sphere((x*a, y*a, z*a), size=r_atom, color=color_atom, opacity=0.3))
                
                # Flächenmitten (Face centered atoms) - nur wenn innerhalb der Grenzen
                if x < range_cells and y < range_cells:
                    atoms.append(sphere(((x+0.5)*a, (y+0.5)*a, z*a), size=r_atom, color=color_atom, opacity=0.3))
                if x < range_cells and z < range_cells:
                    atoms.append(sphere(((x+0.5)*a, y*a, (z+0.5)*a), size=r_atom, color=color_atom, opacity=0.3))
                if y < range_cells and z < range_cells:
                    atoms.append(sphere((x*a, (y+0.5)*a, (z+0.5)*a), size=r_atom, color=color_atom, opacity=0.3))

    # --- LÜCKEN (VOIDS) GENERIEREN ---
    # Oktaederlücken in FCC liegen bei:
    # 1. Zentrum der Zelle: (0.5, 0.5, 0.5)
    # 2. Mitten der Kanten: (0.5, 0, 0), (0, 0.5, 0), etc.
    
    target_void_pos = (0.5*a, 0.5*a, 0.5*a) # Speichern für das Drahtgitter später
    
    for x in range(range_cells):
        for y in range(range_cells):
            for z in range(range_cells):
                # Zentrum der Zelle
                void_pos = ((x+0.5)*a, (y+0.5)*a, (z+0.5)*a)
                voids.append(sphere(void_pos, size=r_void, color=color_void, opacity=0.9))
                
                # Kantenmitten (Edge centers)
                # x-Kanten
                voids.append(sphere(((x+0.5)*a, y*a, z*a), size=r_void, color=color_void, opacity=0.9))
                # y-Kanten
                voids.append(sphere((x*a, (y+0.5)*a, z*a), size=r_void, color=color_void, opacity=0.9))
                # z-Kanten
                voids.append(sphere((x*a, y*a, (z+0.5)*a), size=r_void, color=color_void, opacity=0.9))

    # --- OKTAEDER-KÄFIG (Geometrie-Beweis) ---
    # Wir zeichnen für die erste zentrale Lücke das Oktaeder ein,
    # indem wir die 6 Nachbaratome verbinden.
    # Lücke bei (0.5, 0.5, 0.5)
    # Nachbarn sind die Flächenmitten der angrenzenden Flächen.
    
    center = target_void_pos
    # Die 6 Nachbarn einer Lücke im Zentrum (0.5, 0.5, 0.5) sind:
    neighbors = [
        (0.5*a, 0.5*a, 0.0),   (0.5*a, 0.5*a, 1.0*a), # Oben/Unten
        (0.5*a, 0.0, 0.5*a),   (0.5*a, 1.0*a, 0.5*a), # Vorne/Hinten
        (0.0, 0.5*a, 0.5*a),   (1.0*a, 0.5*a, 0.5*a)  # Links/Rechts
    ]
    
    cage_lines = []
    # Alle Nachbarn miteinander verbinden (Drahtgitter)
    for p1 in neighbors:
        for p2 in neighbors:
            # Verbinde nur, wenn Distanz = a/sqrt(2) (Kanten des Oktaeders)
            # Distanzquadrat muss 0.5*a^2 sein
            d2 = (p1[0]-p2[0])**2 + (p1[1]-p2[1])**2 + (p1[2]-p2[2])**2
            if abs(d2 - 0.5*a**2) < 0.01:
                cage_lines.append(line3d([p1, p2], color=color_cage, thickness=3))

    # --- ZUSAMMENFÜHREN ---
    print("Erstelle Szene...")
    scene = sum(atoms) + sum(voids) + sum(cage_lines)
    
    # --- ANIMATION ERSTELLEN ---
    # Wir erstellen Frames durch Rotation um die Z-Achse
    frames = []
    num_frames = 30 # Anzahl der Bilder (höher = flüssiger, aber rechenintensiver)
    
    print("Rendere Frames für Animation...")
    for i in range(num_frames):
        angle = (2 * pi * i) / num_frames
        # Rotation um Vektor (1,1,1) sieht oft dynamischer aus, hier z-Achse für Stabilität
        frames.append(scene.rotateZ(angle))
        
    # Animationsobjekt erstellen
    anim = animate(frames)
    
    return anim, scene

## Ausführung und Visualisierung

In [3]:
# Ausführen
anim, static_scene = create_fcc_animation()

Erstelle Szene...
Rendere Frames für Animation...


### Option 1: Interaktives 3D-Modell (empfohlen)
Zeigt das statische Modell, das man mit der Maus drehen kann.

In [4]:
# Option 1: Interaktives 3D-Modell direkt anzeigen (beste Qualität im Browser)
# Zeigt das statische Modell, das man mit der Maus drehen kann.
print("Zeige interaktives 3D-Modell (mit der Maus drehbar):")
static_scene.show(viewer='threejs', theme='dark', frame=False)

Zeige interaktives 3D-Modell (mit der Maus drehbar):


Graphics3d Object

### Option 2: Animation als GIF
Erfordert ImageMagick auf dem System.

In [5]:
# Option 2: Animation als GIF rendern (erfordert ImageMagick auf dem System)
# Um dies zu nutzen, entfernen Sie das Kommentarzeichen (#) unten:
# anim.show(delay=15)